# SemEval-2027 ComPartMent – Subtask A: English Noun Compounds

Notebook này triển khai quy trình EDA, chia validation theo **Compound**, và ba nhóm phương pháp theo kế hoạch nghiên cứu:

1. **Static distributional vectors**: đặc trưng vector tĩnh và hồi quy Ridge ở mức compound.
2. **BERT frozen (không fine-tune)**: trích xuất embedding ngữ cảnh và độ tương đồng giữa compound, modifier, head; sau đó học Ridge trên đặc trưng đã trích xuất.
3. **DeBERTa fine-tuning**: fine-tune encoder với hai đầu ra hồi quy `ModAvg` và `HeadAvg`.

Có thêm **mean baseline** làm mốc. Chưa triển khai LLM, stacking hay các hướng mở rộng.

> **Phạm vi cần lưu ý:** PDF và TSV đi kèm là **Subtask A – English Noun Compounds (EN-NN)**, không phải English Particle Verbs (EN-PV). Notebook này vì vậy dự đoán hai điểm `ModAvg` và `HeadAvg` cho mỗi câu. Nếu nhóm quay lại EN-PV, cần đổi schema và mô hình đầu ra.

## Nguyên tắc thực nghiệm

- Chia theo `Compound`: mọi câu của một compound phải nằm cùng một fold.
- Stratify theo bin của điểm `ModAvg` trung bình ở mức compound, theo kế hoạch PDF.
- Không dùng `ContextID`, thứ tự dòng hoặc nhãn validation/test làm đặc trưng.
- `ModStd` và `HeadStd` là độ bất đồng giữa người đánh giá cho **một câu**; không phải context variance metric.
- Báo cáo riêng cho modifier và head: Spearman, context-variance correlation và RMSE.
- Mặc định chạy một fold để kiểm tra pipeline; tăng lên 5 folds × 3 seeds khi đã ổn định.

## Cài thư viện

Nếu thiếu thư viện, mở cell cài đặt bên dưới và chạy một lần. BERT/DeBERTa cần internet ở lần đầu để tải checkpoint từ Hugging Face.

In [ ]:
# Chỉ chạy nếu môi trường còn thiếu package.
# %pip install pandas numpy matplotlib seaborn scipy scikit-learn gensim transformers torch accelerate

## 0. Imports và cấu hình

In [ ]:
import os, re, gc, json, math, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import spearmanr
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

RANDOM_SEEDS = [42, 2026, 2027]
N_SPLITS = 5
ACTIVE_SEED = 42
ACTIVE_FOLD = 0

# Notebook thử tìm dữ liệu ở các vị trí phổ biến.
DATA_CANDIDATES = [
    Path("data/en-nn-train.tsv"),
    Path("en-nn-train.tsv"),
    Path("/mnt/data/en-nn-train.tsv"),
]
OUTPUT_DIR = Path("outputs_en_nn")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COLS = ["ModAvg", "HeadAvg"]
STD_COLS = ["ModStd", "HeadStd"]
EXPECTED_COLUMNS = [
    "ContextID", "Compound", "Mod", "Head",
    "ModAvg", "ModStd", "HeadAvg", "HeadStd", "Context"
]

# Hyperparameters; điều chỉnh sau khi có baseline.
STATIC_EMBEDDING_NAME = "glove-wiki-gigaword-100"
FROZEN_BERT_NAME = "bert-base-uncased"
FROZEN_BERT_LAYER = 4       # hidden_states[4] = đầu ra Transformer layer thứ 4
FINETUNE_MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 256
BATCH_SIZE = 8
EPOCHS = 5
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
PATIENCE = 2

print("Output directory:", OUTPUT_DIR.resolve())

## 1. Đọc dữ liệu

In [ ]:
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy en-nn-train.tsv. Hãy đặt file trong data/ hoặc cùng thư mục notebook, "
        "hoặc sửa DATA_CANDIDATES."
    )

df = pd.read_csv(DATA_PATH, sep="\t")
missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df.columns))
if missing_columns:
    raise ValueError(f"TSV thiếu cột bắt buộc: {missing_columns}")

# Giữ nguyên các cột gốc; chỉ chuyển kiểu cho các cột số.
for c in LABEL_COLS + STD_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["Compound"] = df["Compound"].astype(str).str.strip()
df["Mod"] = df["Mod"].astype(str).str.strip()
df["Head"] = df["Head"].astype(str).str.strip()
df["Context"] = df["Context"].astype(str)

print("File:", DATA_PATH)
print("Shape:", df.shape)
display(df.head())

## 2. Kiểm tra chất lượng dữ liệu

In [ ]:
quality = pd.DataFrame({
    "check": [
        "rows", "unique_compounds", "duplicate_ContextID", "duplicate_full_rows",
        "missing_any", "invalid_ModAvg", "invalid_HeadAvg",
        "invalid_ModStd", "invalid_HeadStd"
    ],
    "value": [
        len(df), df["Compound"].nunique(), df["ContextID"].duplicated().sum(),
        df.duplicated().sum(), df.isna().any(axis=1).sum(),
        ((df["ModAvg"] < 0) | (df["ModAvg"] > 5)).sum(),
        ((df["HeadAvg"] < 0) | (df["HeadAvg"] > 5)).sum(),
        (df["ModStd"] < 0).sum(), (df["HeadStd"] < 0).sum()
    ]
})
display(quality)
print("Columns:", df.columns.tolist())
print("Mod/Head multiword counts:",
      int(df["Mod"].str.contains(r"\s").sum()),
      int(df["Head"].str.contains(r"\s").sum()))

## 3. EDA: phân phối nhãn và số ngữ cảnh theo compound

In [ ]:
print("Unique compounds:", df["Compound"].nunique())
print("Rows per compound:")
display(df.groupby("Compound").size().describe().to_frame("n_contexts"))

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
for ax, col, title in zip(
    axes.flat,
    ["ModAvg", "HeadAvg", "ModStd", "HeadStd"],
    ["Modifier mean rating", "Head mean rating", "Modifier annotator Std", "Head annotator Std"]
):
    ax.hist(df[col].dropna(), bins=20)
    ax.set_title(title)
    ax.set_xlabel(col)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df["ModAvg"], df["HeadAvg"], alpha=0.25, s=18)
ax.set_title("ModAvg vs HeadAvg across contexts")
plt.show()

## 4. EDA ở mức type (Compound)

In [ ]:
target_summary = (
    df.groupby("Compound")
      .agg(
          n_contexts=("ContextID", "size"),
          Mod_mean=("ModAvg", "mean"),
          Head_mean=("HeadAvg", "mean"),
          Mod_annotator_std_mean=("ModStd", "mean"),
          Head_annotator_std_mean=("HeadStd", "mean"),
      )
      .reset_index()
)
# Độ lệch chuẩn của điểm gold qua các câu thuộc cùng compound (population std).
mod_ctx_std = df.groupby("Compound")["ModAvg"].std(ddof=0).rename("Mod_context_std_pop")
head_ctx_std = df.groupby("Compound")["HeadAvg"].std(ddof=0).rename("Head_context_std_pop")
target_summary = target_summary.merge(mod_ctx_std, on="Compound").merge(head_ctx_std, on="Compound")

display(target_summary.sort_values("n_contexts", ascending=False).head(15))
print("Target-level averages:")
display(target_summary[["Mod_mean", "Head_mean", "Mod_context_std_pop", "Head_context_std_pop"]].describe())

### Cách đọc các độ lệch chuẩn

- `ModStd` / `HeadStd`: mức bất đồng giữa annotator **trong một câu**.
- `Mod_context_std_pop` / `Head_context_std_pop`: độ lệch chuẩn của các điểm `Avg` **giữa nhiều câu thuộc cùng một Compound**. Đây mới là đại lượng dùng trong context-variance correlation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].hist(target_summary["Mod_context_std_pop"].dropna(), bins=20)
axes[0].set_title("Across-context Std: Modifier")
axes[1].hist(target_summary["Head_context_std_pop"].dropna(), bins=20)
axes[1].set_title("Across-context Std: Head")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(target_summary["Mod_mean"], target_summary["Mod_context_std_pop"], alpha=0.65)
axes[0].set_title("Modifier: type mean vs context variability")
axes[1].scatter(target_summary["Head_mean"], target_summary["Head_context_std_pop"], alpha=0.65)
axes[1].set_title("Head: type mean vs context variability")
plt.tight_layout()
plt.show()

## 5. Tìm vị trí compound / modifier / head trong câu

In [ ]:
# Dữ liệu có trường hợp head được biến đổi số nhiều trong Context:
# ví dụ Compound='account book', Context chứa 'account books'.
# Ta dò cụm Mod + Head liền nhau, cho phép một số dạng số nhiều tiếng Anh thông dụng.
IRREGULAR_PLURALS = {
    "man": ["men"], "woman": ["women"], "child": ["children"],
    "person": ["people"], "mouse": ["mice"], "goose": ["geese"],
    "tooth": ["teeth"], "foot": ["feet"],
}

def word_forms(word):
    w = str(word).strip().lower()
    forms = {w}
    if not w:
        return []
    forms.update(IRREGULAR_PLURALS.get(w, []))
    if w.endswith("y") and len(w) > 1 and w[-2] not in "aeiou":
        forms.add(w[:-1] + "ies")
    elif w.endswith(("s", "x", "z", "ch", "sh")):
        forms.add(w + "es")
    else:
        forms.add(w + "s")
    if w.endswith("fe"):
        forms.add(w[:-2] + "ves")
    elif w.endswith("f"):
        forms.add(w[:-1] + "ves")
    return sorted(forms, key=len, reverse=True)

def make_alt(words):
    return "(?:" + "|".join(re.escape(x) for x in words) + ")"

def find_compound_span(context, mod, head):
    mod_alt = make_alt(word_forms(mod))
    head_alt = make_alt(word_forms(head))
    pattern = re.compile(rf"(?<![A-Za-z0-9_])(?P<mod>{mod_alt})(?P<gap>\s+)(?P<head>{head_alt})(?![A-Za-z0-9_])", re.I)
    m = pattern.search(str(context))
    if m is None:
        return None
    return {
        "start": m.start(), "end": m.end(),
        "mod_start": m.start("mod"), "mod_end": m.end("mod"),
        "head_start": m.start("head"), "head_end": m.end("head"),
        "matched_text": m.group(0)
    }

spans = [find_compound_span(r.Context, r.Mod, r.Head) for r in df.itertuples()]
df["_span"] = spans
n_unmatched = sum(x is None for x in spans)
print(f"Matched compound span: {len(df)-n_unmatched}/{len(df)} ({(len(df)-n_unmatched)/len(df):.2%})")
if n_unmatched:
    display(df.loc[df["_span"].isna(), ["ContextID", "Compound", "Mod", "Head", "Context"]].head(20))
else:
    print("Tất cả các dòng đều tìm được span Mod + Head liền nhau.")

display(df[["Compound", "Mod", "Head", "Context"]].head(8))

**Lưu ý:** bộ dò trên xử lý một số biến thể số nhiều phổ biến; không thay thế parser ngữ pháp tổng quát. Trước khi huấn luyện, cần xem thủ công các trường hợp không tìm được span. Các mô hình BERT trong notebook cần span để đánh dấu đúng thành phần trong câu.

## 6. Chia dữ liệu theo target: StratifiedGroupKFold

In [ ]:
def build_strata(df, n_bins=5):
    # Một nhãn phân tầng cho mỗi Compound, dựa trên trung bình ModAvg của Compound đó.
    type_mod_mean = df.groupby("Compound")["ModAvg"].mean()
    q = min(n_bins, type_mod_mean.nunique())
    bins = pd.qcut(type_mod_mean, q=q, labels=False, duplicates="drop")
    return df["Compound"].map(bins).astype(int)

def make_group_folds(data, seeds=RANDOM_SEEDS, n_splits=N_SPLITS):
    strata = build_strata(data)
    all_folds = {}
    for seed in seeds:
        splitter = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        seed_folds = []
        for fold, (train_idx, val_idx) in enumerate(
            splitter.split(data, y=strata, groups=data["Compound"])
        ):
            train_groups = set(data.iloc[train_idx]["Compound"])
            val_groups = set(data.iloc[val_idx]["Compound"])
            assert train_groups.isdisjoint(val_groups), "Target leakage detected!"
            seed_folds.append((train_idx, val_idx))
        all_folds[seed] = seed_folds
    return all_folds

folds = make_group_folds(df)
train_idx, val_idx = folds[ACTIVE_SEED][ACTIVE_FOLD]
train_df = df.iloc[train_idx].copy().reset_index(drop=True)
val_df = df.iloc[val_idx].copy().reset_index(drop=True)

print("Seed / fold:", ACTIVE_SEED, ACTIVE_FOLD)
print("Train rows / compounds:", len(train_df), train_df["Compound"].nunique())
print("Valid rows / compounds:", len(val_df), val_df["Compound"].nunique())
print("Overlap compounds:", len(set(train_df.Compound) & set(val_df.Compound)))

display(pd.DataFrame([
    {"split": "train", "rows": len(train_df), "compounds": train_df.Compound.nunique()},
    {"split": "validation", "rows": len(val_df), "compounds": val_df.Compound.nunique()},
]))

# Lưu manifest để tất cả mô hình dùng đúng cùng một split.
manifest = pd.DataFrame({"row_index": np.arange(len(df)), "Compound": df["Compound"], "ContextID": df["ContextID"]})
manifest["split"] = "train"
manifest.loc[val_idx, "split"] = "validation"
manifest["seed"] = ACTIVE_SEED
manifest["fold"] = ACTIVE_FOLD
manifest.to_csv(OUTPUT_DIR / "split_manifest.csv", index=False)

## 7. Evaluator: ba metric của ComPartMent

In [ ]:
def safe_spearman(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    mask = np.isfinite(a) & np.isfinite(b)
    a, b = a[mask], b[mask]
    if len(a) < 2 or np.unique(a).size < 2 or np.unique(b).size < 2:
        return np.nan
    return float(spearmanr(a, b).statistic)

def evaluate_one_target(data, pred, gold_col, target_col="Compound"):
    gold = data[gold_col].to_numpy(dtype=float)
    pred = np.asarray(pred, dtype=float)
    rmse = float(np.sqrt(mean_squared_error(gold, pred)))
    rho = safe_spearman(gold, pred)

    tmp = pd.DataFrame({target_col: data[target_col].values, "gold": gold, "pred": pred})
    grouped = tmp.groupby(target_col).agg(
        gold_std=("gold", lambda x: np.std(x, ddof=0)),
        pred_std=("pred", lambda x: np.std(x, ddof=0)),
        n=("gold", "size")
    )
    grouped = grouped[grouped["n"] >= 2].copy()
    # Chuẩn hóa sai số số học rất nhỏ thành 0; tránh Spearman giả khi prediction hằng số.
    grouped.loc[grouped["pred_std"] <= 1e-12, "pred_std"] = 0.0
    grouped.loc[grouped["gold_std"] <= 1e-12, "gold_std"] = 0.0
    context_var_rho = safe_spearman(grouped["gold_std"], grouped["pred_std"])
    zero_pred_std = int((grouped["pred_std"] <= 1e-12).sum())
    return {
        "Spearman": rho,
        "Context_variance_rho": context_var_rho,
        "RMSE": rmse,
        "n_compounds_for_variance": len(grouped),
        "zero_pred_std_compounds": zero_pred_std,
    }

def evaluate_predictions(data, pred_mod, pred_head, model_name="model"):
    rows = []
    for label, pred in zip(LABEL_COLS, [pred_mod, pred_head]):
        metrics = evaluate_one_target(data, pred, label)
        rows.append({"model": model_name, "target": label, **metrics})
    return pd.DataFrame(rows)

## 8. B0 – Mean baseline

In [ ]:
mean_mod = train_df["ModAvg"].mean()
mean_head = train_df["HeadAvg"].mean()
pred_b0_mod = np.full(len(val_df), mean_mod)
pred_b0_head = np.full(len(val_df), mean_head)
metrics_b0 = evaluate_predictions(val_df, pred_b0_mod, pred_b0_head, "B0_mean")
display(metrics_b0)

## 9. B1 – Static distributional vectors

Phương pháp theo PDF: biểu diễn compound và các thành phần bằng vector tĩnh, rồi dùng cosine similarity làm đặc trưng cho mô hình hồi quy.

**Cách triển khai trong notebook:** tải GloVe 100 chiều; lấy vector tĩnh của `Mod` và `Head`; xây dựng một *type-level context profile* bằng cách trung bình vector của các từ xung quanh compound trong các câu thuộc cùng compound (sau khi bỏ span compound). Sau đó tính cosine giữa profile với Mod, Head, và giữa Mod–Head; Ridge dự đoán điểm trung bình ở mức compound rồi broadcast điểm đó cho các câu của compound.

Đây là **đặc trưng không giám sát có sử dụng văn bản của các câu validation thuộc compound đó**, nhưng tuyệt đối không sử dụng nhãn validation. Đó là thiết lập transductive được đề xuất trong PDF; ghi rõ khi báo cáo. Dự đoán B1 không thay đổi giữa các câu cùng compound, nên context-variance correlation thường không xác định và đây là hạn chế có chủ đích của baseline type-level.

In [ ]:
# Cần gensim và internet ở lần đầu tải GloVe.
# Nếu đã có KeyedVectors cục bộ, có thể bỏ qua download và load bằng KeyedVectors.load(...).
try:
    import gensim.downloader as api
    kv = api.load(STATIC_EMBEDDING_NAME)
    print("Loaded:", STATIC_EMBEDDING_NAME, "| vector size:", kv.vector_size)
except Exception as e:
    kv = None
    print("Không tải được GloVe. Kiểm tra internet/gensim hoặc dùng KeyedVectors cục bộ.")
    print(type(e).__name__, str(e)[:500])

In [ ]:
WORD_RE = re.compile(r"[A-Za-z]+(?:['’][A-Za-z]+)?")

def get_static_vector(text, kv):
    vecs = []
    for token in WORD_RE.findall(str(text).lower()):
        if token in kv:
            vecs.append(kv[token])
    if not vecs:
        return None
    return np.mean(np.vstack(vecs), axis=0)

def cosine_np(a, b):
    if a is None or b is None:
        return np.nan
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na == 0 or nb == 0:
        return np.nan
    return float(np.dot(a, b) / (na * nb))

def static_type_features(data, kv):
    rows = []
    for compound, group in data.groupby("Compound", sort=False):
        mod = group["Mod"].iloc[0]
        head = group["Head"].iloc[0]
        mod_vec = get_static_vector(mod, kv)
        head_vec = get_static_vector(head, kv)
        context_vecs = []
        for _, r in group.iterrows():
            span = r["_span"]
            if span is None:
                continue
            left = r["Context"][:span["start"]]
            right = r["Context"][span["end"]:]
            v = get_static_vector(left + " " + right, kv)
            if v is not None:
                context_vecs.append(v)
        profile = np.mean(np.vstack(context_vecs), axis=0) if context_vecs else None
        rows.append({
            "Compound": compound,
            "cos_profile_mod": cosine_np(profile, mod_vec),
            "cos_profile_head": cosine_np(profile, head_vec),
            "cos_mod_head": cosine_np(mod_vec, head_vec),
            "profile_coverage": len(context_vecs) / max(1, len(group)),
        })
    return pd.DataFrame(rows)

if kv is not None:
    static_features = static_type_features(df, kv)
    display(static_features.head())
else:
    static_features = None

In [ ]:
if static_features is not None:
    # Dùng nhãn train ở mức type: trung bình nhãn qua các câu của compound.
    type_labels = train_df.groupby("Compound")[LABEL_COLS].mean().reset_index()
    train_types = type_labels.merge(static_features, on="Compound", how="left")
    val_types = val_df[["Compound"]].drop_duplicates().merge(static_features, on="Compound", how="left")

    feature_cols = ["cos_profile_mod", "cos_profile_head", "cos_mod_head", "profile_coverage"]
    static_reg = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=10.0))
    static_reg.fit(train_types[feature_cols], train_types[LABEL_COLS])
    val_type_pred = static_reg.predict(val_types[feature_cols])
    val_types[["pred_ModAvg", "pred_HeadAvg"]] = val_type_pred
    pred_static_map = val_types.set_index("Compound")[["pred_ModAvg", "pred_HeadAvg"]]
    pred_static = val_df["Compound"].map(pred_static_map["pred_ModAvg"]).to_numpy(), val_df["Compound"].map(pred_static_map["pred_HeadAvg"]).to_numpy()
    pred_static_mod, pred_static_head = pred_static
    metrics_static = evaluate_predictions(val_df, pred_static_mod, pred_static_head, "B1_static_vectors")
    display(metrics_static)
    val_types.to_csv(OUTPUT_DIR / "B1_static_type_predictions.csv", index=False)
else:
    print("Bỏ qua B1 vì chưa tải được static embeddings.")

## 10. B2 – BERT frozen (không fine-tune)

BERT được giữ nguyên trọng số. Ta trích xuất hidden states ở một layer cố định rồi tạo đặc trưng cosine:

- compound span trong câu ↔ modifier span trong câu;
- compound span trong câu ↔ head span trong câu;
- modifier trong câu ↔ modifier khi đứng riêng;
- head trong câu ↔ head khi đứng riêng;
- modifier trong câu ↔ head trong câu.

Sau khi trích xuất, dùng Ridge Regression để học từ các đặc trưng đó. BERT **không được cập nhật trọng số**; chỉ Ridge được huấn luyện.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

In [ ]:
def pool_char_span(hidden, offsets, start, end):
    # hidden: [seq_len, hidden_size], offsets: list[(start,end)]
    ids = [i for i, (a, b) in enumerate(offsets) if b > a and a < end and b > start]
    if not ids:
        return None
    return hidden[ids].mean(dim=0)

def cosine_torch(a, b):
    if a is None or b is None:
        return np.nan
    b = b.to(a.device)  # prototype có thể nằm trên CPU, context embedding có thể nằm trên GPU
    return float(torch.nn.functional.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item())

@torch.no_grad()
def frozen_bert_features(data, model_name=FROZEN_BERT_NAME, layer=FROZEN_BERT_LAYER, batch_size=16):
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModel.from_pretrained(model_name, output_hidden_states=True).to(DEVICE)
    model.eval()

    # Standalone prototype embedding cho các thành phần duy nhất.
    vocab_terms = sorted(set(data["Mod"].astype(str)) | set(data["Head"].astype(str)))
    proto = {}
    for start in range(0, len(vocab_terms), batch_size):
        terms = vocab_terms[start:start + batch_size]
        enc = tokenizer(terms, padding=True, truncation=True, return_offsets_mapping=True, return_tensors="pt")
        offsets = enc.pop("offset_mapping").tolist()
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        out = model(**enc)
        h = out.hidden_states[layer]
        for i, term in enumerate(terms):
            # Chỉ mean-pool token nội dung có offset hợp lệ; bỏ CLS/SEP/PAD.
            valid_ids = [j for j, (a, b) in enumerate(offsets[i]) if b > a]
            proto[term] = h[i][valid_ids].mean(0).cpu() if valid_ids else None

    feature_rows = []
    for start in range(0, len(data), batch_size):
        batch = data.iloc[start:start + batch_size]
        texts = batch["Context"].tolist()
        enc = tokenizer(texts, padding=True, truncation=True, max_length=MAX_LENGTH,
                        return_offsets_mapping=True, return_tensors="pt")
        offsets_batch = enc.pop("offset_mapping").tolist()
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        out = model(**enc)
        h_batch = out.hidden_states[layer]

        for j, (_, row) in enumerate(batch.iterrows()):
            span = row["_span"]
            if span is None:
                feature_rows.append({k: np.nan for k in ["cos_comp_mod", "cos_comp_head", "cos_mod_proto", "cos_head_proto", "cos_mod_head"]})
                continue
            offsets = [tuple(x) for x in offsets_batch[j]]
            h = h_batch[j]
            comp_v = pool_char_span(h, offsets, span["start"], span["end"])
            mod_v = pool_char_span(h, offsets, span["mod_start"], span["mod_end"])
            head_v = pool_char_span(h, offsets, span["head_start"], span["head_end"])
            mod_p = proto.get(row["Mod"])
            head_p = proto.get(row["Head"])
            feature_rows.append({
                "cos_comp_mod": cosine_torch(comp_v, mod_v),
                "cos_comp_head": cosine_torch(comp_v, head_v),
                "cos_mod_proto": cosine_torch(mod_v, mod_p),
                "cos_head_proto": cosine_torch(head_v, head_p),
                "cos_mod_head": cosine_torch(mod_v, head_v),
            })
    del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return pd.DataFrame(feature_rows)

In [ ]:
# Feature extraction chạy trên toàn bộ các câu vì encoder frozen và không dùng nhãn.
# Cache để không phải trích xuất lại ở các lần chạy sau.
FROZEN_CACHE = OUTPUT_DIR / "bert_frozen_features.csv"
if FROZEN_CACHE.exists():
    frozen_features = pd.read_csv(FROZEN_CACHE)
    print("Loaded cache:", FROZEN_CACHE)
else:
    frozen_features = frozen_bert_features(df)
    frozen_features.to_csv(FROZEN_CACHE, index=False)
    print("Saved cache:", FROZEN_CACHE)

display(frozen_features.head())

In [ ]:
feature_cols_frozen = list(frozen_features.columns)
X_train_frozen = frozen_features.iloc[train_idx].reset_index(drop=True)
X_val_frozen = frozen_features.iloc[val_idx].reset_index(drop=True)

frozen_reg = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=10.0))
frozen_reg.fit(X_train_frozen, train_df[LABEL_COLS])
pred_frozen = frozen_reg.predict(X_val_frozen)
pred_frozen_mod, pred_frozen_head = pred_frozen[:, 0], pred_frozen[:, 1]
metrics_frozen = evaluate_predictions(val_df, pred_frozen_mod, pred_frozen_head, "B2_BERT_frozen")
display(metrics_frozen)

## 11. B3 – DeBERTa fine-tuning cho hai đầu ra

Mô hình nhận cặp đầu vào:

`[marked context] [SEP] [compound]`

Trong câu, modifier và head được đánh dấu bằng special tokens `<m>...</m>` và `<h>...</h>`. Encoder DeBERTa được fine-tune và một regression head dự đoán đồng thời `ModAvg`, `HeadAvg`. Đầu ra dùng `5 × sigmoid` để nằm trong khoảng 0–5.

Mặc định chỉ huấn luyện `ACTIVE_FOLD` để kiểm tra pipeline. Khi chạy báo cáo cuối, nên lặp đủ 5 folds và 3 seeds, lưu dự đoán out-of-fold.

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import copy

SPECIAL_MARKERS = ["<m>", "</m>", "<h>", "</h>"]

def mark_context(row):
    s = row["Context"]
    span = row["_span"]
    if span is None:
        raise ValueError(f"Không tìm thấy compound span: {row['Compound']} | {s}")
    return (
        s[:span["mod_start"]] + "<m>" + s[span["mod_start"]:span["mod_end"]] + "</m>" +
        s[span["mod_end"]:span["head_start"]] +
        "<h>" + s[span["head_start"]:span["head_end"]] + "</h>" +
        s[span["head_end"]:]
    )

class CompoundDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=MAX_LENGTH):
        marked = [mark_context(r) for _, r in frame.iterrows()]
        compounds = frame["Compound"].tolist()
        enc = tokenizer(marked, compounds, truncation=True, max_length=max_length,
                        padding="max_length", return_tensors="pt")
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = torch.tensor(frame[LABEL_COLS].to_numpy(dtype=np.float32))
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx]
        }

class DebertaRegressor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.1)
        self.regressor = nn.Linear(hidden, 2)
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return 5.0 * torch.sigmoid(self.regressor(self.dropout(cls)))

In [ ]:
def predict_deberta(model, loader):
    model.eval()
    preds, golds = [], []
    with torch.no_grad():
        for batch in loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            preds.append(model(ids, mask).cpu().numpy())
            golds.append(batch["labels"].numpy())
    return np.vstack(preds), np.vstack(golds)

def run_deberta_fold(train_frame, valid_frame, fold_name="fold0", seed=ACTIVE_SEED):
    # Cố định seed để có thể tái lập lần chạy.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(FINETUNE_MODEL_NAME, use_fast=True)
    tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_MARKERS})

    train_ds = CompoundDataset(train_frame, tokenizer)
    valid_ds = CompoundDataset(valid_frame, tokenizer)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)

    model = DebertaRegressor(FINETUNE_MODEL_NAME).to(DEVICE)
    model.encoder.resize_token_embeddings(len(tokenizer))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    best_score, best_state, best_epoch, bad_epochs = -np.inf, None, 0, 0
    history = []
    for epoch in range(1, EPOCHS + 1):
        model.train()
        losses = []
        for batch in train_loader:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            preds = model(ids, mask)
            loss = F.mse_loss(preds, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            losses.append(loss.item())

        val_pred, val_gold = predict_deberta(model, valid_loader)
        val_metrics = evaluate_predictions(
            valid_frame, val_pred[:, 0], val_pred[:, 1], f"DeBERTa_epoch_{epoch}"
        )
        valid_rhos = val_metrics["Spearman"].dropna()
        score = float(valid_rhos.mean()) if len(valid_rhos) else -float(val_metrics["RMSE"].mean())
        history.append({"epoch": epoch, "train_mse": np.mean(losses), "selection_score": score,
                        "Mod_spearman": val_metrics.loc[0, "Spearman"],
                        "Head_spearman": val_metrics.loc[1, "Spearman"],
                        "Mod_RMSE": val_metrics.loc[0, "RMSE"],
                        "Head_RMSE": val_metrics.loc[1, "RMSE"]})
        print(f"Epoch {epoch}/{EPOCHS} | train MSE={np.mean(losses):.4f} | selection={score:.4f}")

        if score > best_score:
            best_score, best_epoch, bad_epochs = score, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("Early stopping.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    final_pred, final_gold = predict_deberta(model, valid_loader)
    final_metrics = evaluate_predictions(valid_frame, final_pred[:, 0], final_pred[:, 1], "B3_DeBERTa_finetuned")
    hist = pd.DataFrame(history)
    hist.to_csv(OUTPUT_DIR / f"B3_{fold_name}_history.csv", index=False)
    final_out = valid_frame[["ContextID", "Compound", "Context", "ModAvg", "HeadAvg"]].copy()
    final_out["pred_ModAvg"] = final_pred[:, 0]
    final_out["pred_HeadAvg"] = final_pred[:, 1]
    final_out.to_csv(OUTPUT_DIR / f"B3_{fold_name}_predictions.csv", index=False)
    model_path = OUTPUT_DIR / f"B3_{fold_name}_checkpoint"
    model.encoder.save_pretrained(model_path)
    tokenizer.save_pretrained(model_path)
    torch.save(model.regressor.state_dict(), model_path / "regression_head.pt")

    del model, train_loader, valid_loader, train_ds, valid_ds
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return final_metrics, hist, final_pred

In [ ]:
# Chạy DeBERTa fine-tuning. Có thể mất thời gian và cần tải model (~hàng trăm MB).
RUN_FINETUNING = False  # Đổi thành True khi sẵn sàng chạy.

if RUN_FINETUNING:
    metrics_deberta, history_deberta, pred_deberta = run_deberta_fold(
        train_df, val_df, fold_name=f"seed{ACTIVE_SEED}_fold{ACTIVE_FOLD}"
    )
    display(metrics_deberta)
    display(history_deberta)
else:
    print("Chưa chạy fine-tuning. Đổi RUN_FINETUNING=True để bắt đầu.")

## 12. So sánh kết quả các baseline

In [ ]:
metric_frames = [metrics_b0]
if "metrics_static" in globals():
    metric_frames.append(metrics_static)
if "metrics_frozen" in globals():
    metric_frames.append(metrics_frozen)
if "metrics_deberta" in globals():
    metric_frames.append(metrics_deberta)

all_metrics = pd.concat(metric_frames, ignore_index=True)
display(all_metrics)
all_metrics.to_csv(OUTPUT_DIR / "baseline_comparison.csv", index=False)

## 13. Phân tích dự đoán và lưu kết quả

Khi đã chạy một mô hình, hãy kiểm tra các ví dụ sai nhiều nhất. Không dùng `ContextID` làm feature; chỉ giữ nó để truy vết dự đoán.

In [ ]:
# Ví dụ: khi BERT frozen đã chạy, xuất bảng sai số để xem thủ công.
if "pred_frozen" in globals():
    error_analysis = val_df[["ContextID", "Compound", "Mod", "Head", "Context", "ModAvg", "HeadAvg"]].copy()
    error_analysis["pred_ModAvg"] = pred_frozen_mod
    error_analysis["pred_HeadAvg"] = pred_frozen_head
    error_analysis["abs_error_Mod"] = np.abs(error_analysis["ModAvg"] - error_analysis["pred_ModAvg"])
    error_analysis["abs_error_Head"] = np.abs(error_analysis["HeadAvg"] - error_analysis["pred_HeadAvg"])
    display(error_analysis.sort_values("abs_error_Mod", ascending=False).head(20))
    error_analysis.to_csv(OUTPUT_DIR / "B2_error_analysis.csv", index=False)

print("Outputs saved to:", OUTPUT_DIR.resolve())

## 14. Giao thức chạy thực nghiệm đầy đủ

1. Kiểm tra EDA và xác nhận các span được tìm đúng; xem thủ công mọi dòng không match.
2. Chạy B0 và B1, ghi rõ B1 dùng type-level context profile từ văn bản không nhãn của các câu cùng target (transductive unsupervised features).
3. Chạy B2 để trích xuất/cached frozen BERT features; sau đó Ridge được fit trong mỗi fold, không cập nhật BERT.
4. Chạy B3 fine-tuning. Bắt đầu với một fold; sau khi pipeline ổn định, chạy đủ 5 folds × 3 seeds.
5. So sánh mô hình bằng cùng split. Báo cáo mean ± std giữa folds/seeds; không chọn mô hình chỉ dựa trên một lần chia.
6. Ghi cả Spearman, context-variance correlation và RMSE cho Mod/Head. Với B0/B1, context-variance correlation có thể NaN vì dự đoán không đổi trong cùng compound; đây là đặc tính của baseline, không phải lỗi code.

### Các giới hạn cần ghi trong báo cáo

- Mẫu dữ liệu ít (444 compounds), nên Transformer có nguy cơ overfit; dùng early stopping và nhiều seed.
- `ModStd`/`HeadStd` chưa được dùng làm trọng số loss trong notebook này; trước hết giữ baseline đơn giản.
- Quy tắc dò span hiện xử lý cụm hai thành phần liền nhau và biến thể số nhiều thông dụng; trường hợp không match phải rà soát.
- Frozen BERT features phụ thuộc layer được chọn; có thể thử layer 2/4/6 trong ablation sau.
- B1 dùng thông tin văn bản của validation/test compound nhưng không dùng nhãn. Đây là lựa chọn transductive cần nêu rõ và kiểm tra quy định của cuộc thi trước khi nộp.
- Tập validation nội bộ không phải test chính thức của ban tổ chức.